# Flow matching for simple distributions

## Prepare toy data

Generate target distribution data (two gaussians) and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

source_data = np.random.randn(1000, 2)

target_data = np.vstack([
    np.random.multivariate_normal(np.array([10, 5]), np.diag([2, 1]), 500),
    np.random.multivariate_normal(np.array([10, -5]), np.diag([2, 1]), 500)
])

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Source Data",
        marker=dict(color="blue", size=6, opacity=0.7),
    )
)

fig.add_trace(
    go.Scatter(
        x=target_data[:, 0],
        y=target_data[:, 1],
        mode="markers",
        name="Target Data",
        marker=dict(color="orange", size=6, opacity=0.7),
    )
)

fig.update_layout(
    title="Source vs Target Distributions",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[-5, 15], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0, pad=0),
    template="plotly_white",
)

fig.show()

## Basic flow matching

Independent coupling between source and target data points

In [ ]:
n_pairs = 1000

src_idx = np.random.randint(0, source_data.shape[0], size=n_pairs)
tgt_idx = np.random.randint(0, target_data.shape[0], size=n_pairs)

couplings = [(source_data[i], target_data[j]) for i, j in zip(src_idx, tgt_idx)]

Visualize the generated couplings

In [ ]:
x_lines, y_lines = [], []
for src, tgt in couplings:
    x_lines.extend([src[0], tgt[0], None])
    y_lines.extend([src[1], tgt[1], None])

fig_pairs = go.Figure()

fig_pairs.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Source Data",
        marker=dict(color="blue", size=6, opacity=0.7),
    )
)

fig_pairs.add_trace(
    go.Scatter(
        x=target_data[:, 0],
        y=target_data[:, 1],
        mode="markers",
        name="Target Data",
        marker=dict(color="orange", size=6, opacity=0.7),
    )
)

fig_pairs.add_trace(
    go.Scatter(
        x=x_lines,
        y=y_lines,
        mode="lines",
        name="Couplings",
        line=dict(color="rgba(120,120,120,0.25)", width=1),
        hoverinfo="skip",
    )
)

fig_pairs.update_layout(
    title="Independent Couplings",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[-5, 15], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0, pad=0),
    template="plotly_white",
)

fig_pairs.show()

Define a simple MLP with ReLU activations as the velocity field estimation network.

In [ ]:
import torch
import torch.nn as nn

class FlowMLP(nn.Module):
    def __init__(self, input_output_dim: int, hidden_dim: int, num_blocks: int):
        super().__init__()
        if num_blocks < 1:
            raise ValueError("num_blocks must be >= 1")

        layers = []
        in_dim = input_output_dim

        for _ in range(num_blocks):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        layers.append(nn.Linear(hidden_dim, input_output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

Train the network through backpropagation

In [ ]:
# Prepare training tensors from existing couplings
src_tensor = torch.tensor(np.stack([src for src, _ in couplings]), dtype=torch.float32)
tgt_tensor = torch.tensor(np.stack([tgt for _, tgt in couplings]), dtype=torch.float32)

# Instantiate model for 2D data
model = FlowMLP(input_output_dim=source_data.shape[1], hidden_dim=128, num_blocks=3)

# Training setup
batch_size = 64
num_epochs = 200
learning_rate = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

# Train
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0

    for i in range(0, src_tensor.shape[0], batch_size):
        src_batch = src_tensor[i:i + batch_size]
        tgt_batch = tgt_tensor[i:i + batch_size]

        # Sample t ~ Uniform[0, 1] for each pair in minibatch
        t = torch.rand(src_batch.shape[0], 1)

        # Linear interpolation x_t = (1-t) * src + t * tgt
        x_t = (1.0 - t) * src_batch + t * tgt_batch

        # Target velocity vector: tgt - src
        v_target = tgt_batch - src_batch

        # Predict and optimize
        v_pred = model(x_t)
        loss = criterion(v_pred, v_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * src_batch.shape[0]

    epoch_loss /= src_tensor.shape[0]
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{num_epochs} - Loss: {epoch_loss:.6f}")

Visualize the velocity field we have learned

In [ ]:
import plotly.figure_factory as ff

# Evaluate learned velocity field on a 2D grid and visualize vectors
x_min, x_max = -5, 15
y_min, y_max = -10, 10
n_grid = 25
scale = 0.05  # arrow length scaling for visualization

xg = np.linspace(x_min, x_max, n_grid)
yg = np.linspace(y_min, y_max, n_grid)
X, Y = np.meshgrid(xg, yg)
grid_points = np.stack([X.ravel(), Y.ravel()], axis=1)

device = next(model.parameters()).device
model.eval()
with torch.no_grad():
    v_grid = model(torch.tensor(grid_points, dtype=torch.float32, device=device)).cpu().numpy()

# Build line segments for arrows: (x, y) -> (x + scale*vx, y + scale*vy)
x_lines_field, y_lines_field = [], []
for (x0, y0), (vx, vy) in zip(grid_points, v_grid):
    x_lines_field.extend([x0, x0 + scale * vx, None])
    y_lines_field.extend([y0, y0 + scale * vy, None])

fig_field = go.Figure()

quiver = ff.create_quiver(
    grid_points[:, 0],
    grid_points[:, 1],
    scale * v_grid[:, 0],
    scale * v_grid[:, 1],
    scale=1.0,
    arrow_scale=0.35,  # pointy arrow heads
    line=dict(color="rgba(20,20,20,0.55)", width=1),
    name="Predicted velocity vectors",
)

fig_field.add_trace(
    go.Scatter(
        x=source_data[:, 0],
        y=source_data[:, 1],
        mode="markers",
        name="Source Data",
        marker=dict(color="blue", size=6, opacity=0.7),
    )
)

fig_field.add_trace(
    go.Scatter(
        x=target_data[:, 0],
        y=target_data[:, 1],
        mode="markers",
        name="Target Data",
        marker=dict(color="orange", size=6, opacity=0.7),
    )
)

for trace in quiver.data:
    fig_field.add_trace(trace)

fig_field.update_layout(
    title="Learned Velocity Field",
    width=600,
    height=600,
    xaxis=dict(title="X1", range=[x_min, x_max], constrain="domain", fixedrange=True),
    yaxis=dict(title="X2", scaleanchor="x"),
    legend=dict(orientation="h", yanchor="top", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0, pad=0),
    template="plotly_white",
)

fig_field.show()